## Linear regression
- **For each menu in store** -> linear regression
- **Number of variables** = 


In [35]:
# Import libraries
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


In [36]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']


# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']


    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission.csv")

# Prediction result
all_preds = []


In [37]:
def train_models_by_store_menu(df: pd.DataFrame, degree: int = 2):
    """
    Create and train a separate polynomial regression model for each store_menu_id,
    using the date converted to ordinal and its polynomial features up to the given degree.

    Parameters:
    - df (pd.DataFrame): Input dataframe containing 'store', 'menu', 'date', and 'sales' columns.
    - degree (int): Degree of the polynomial features (default=2 for quadratic).

    Returns:
    - models_dict (dict): Mapping from store_menu_id to its fitted Pipeline model.
    """
    # Ensure date column is datetime type
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])
    
    models_dict = {}
    
    # Train one polynomial-regression pipeline per store_menu_id
    for sm_id, group in df.groupby('store_menu_id'):
        # Prepare feature matrix X and target y
        X = group[['date_ordinal']]
        y = group['sales']
        
        # Build a pipeline: generate polynomial features, then fit linear regression
        pipeline = Pipeline([
            ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
            ('regressor', LinearRegression())
        ])
        
        # Train the model
        pipeline.fit(X, y)
        
        # Store the trained pipeline
        models_dict[sm_id] = pipeline
    
    return models_dict

In [ ]:
def iterative_forecast_for_store_menu(
    store_menu_id: str,
    current_train: pd.DataFrame,
    val_df: pd.DataFrame,
    submission: pd.DataFrame,
    block_idx: int,
    degrees: range
):
    """
    Train polynomial models for one block and one store_menu_id,
    pick best degree, forecast next 7 days, and append to submission.
    """
    # prepare features
    current_train['date_ordinal'] = current_train['date'].map(datetime.toordinal)
    val_df['date_ordinal']     = val_df['date'].map(datetime.toordinal)

    X_train, y_train = current_train[['date_ordinal']], current_train['sales']
    X_val,   y_val   = val_df[['date_ordinal']],      val_df['sales']

    # find best degree
    best_loss, best_model, best_deg = float('inf'), None, None
    for deg in degrees:
        model = Pipeline([
            ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
            ('lr', LinearRegression())
        ])
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        loss  = mean_squared_error(y_val, preds)
        if loss < best_loss:
            best_loss, best_model, best_deg = loss, model, deg

    # forecast next 7 days
    last_date = val_df['date'].max()
    future_dates = [last_date + timedelta(days=d) for d in range(1, 8)]
    X_future = pd.DataFrame({
        'date_ordinal': [d.toordinal() for d in future_dates]
    })
    future_preds = best_model.predict(X_future)

    # fill into submission
    for d, pred in enumerate(future_preds, start=1):
        idx = f"TEST_{block_idx:02d}+{d}일"
        submission.loc[idx, store_menu_id] = float(pred)

    # return forecasted rows for retraining
    df_pred = pd.DataFrame({
        'date': future_dates,
        'sales': future_preds,
        'store_menu_id': store_menu_id
    })
    df_val  = val_df[['date', 'sales', 'store_menu_id']]
    return pd.concat([current_train, df_val, df_pred], ignore_index=True), submission

def iterative_forecast_all(
    train_data: pd.DataFrame,
    test_data_list: list[pd.DataFrame],
    submission: pd.DataFrame,
    degrees: range = range(1, 11)
) -> pd.DataFrame:
    """
    Loop over all store_menu_id and all test blocks,
    perform iterative forecasting and fill submission.
    """
    # ensure datetime and store_menu_id column
    for df in [train_data] + test_data_list:
        if not pd.api.types.is_datetime64_any_dtype(df['date']):
            df['date'] = pd.to_datetime(df['date'])
        if 'store_menu_id' not in df:
            df['store_menu_id'] = df['store'] + '_' + df['menu']

    # get all unique store_menu_ids from submission columns
    all_ids = list(submission.columns)

    # for each store_menu_id
    for sm_id in all_ids:
        # initialize training slice
        current_train = train_data[train_data['store_menu_id'] == sm_id].copy()

        # for each test block
        for i, test_df in enumerate(test_data_list):
            val_df = test_df[test_df['store_menu_id'] == sm_id].copy()

            # skip if no data in this block
            if val_df.empty:
                continue

            # train, forecast, and update train & submission
            current_train, submission = iterative_forecast_for_store_menu(
                store_menu_id=sm_id,
                current_train=current_train,
                val_df=val_df,
                submission=submission,
                block_idx=i,
                degrees=degrees
            )

    return submission


In [39]:
# test_data_0 ~ test_data_9 리스트로 묶기
test_list = [test_data_0, test_data_1, test_data_2, test_data_3, 
             test_data_4, test_data_5, test_data_6, test_data_7, 
             test_data_8, test_data_9]

# 전체 forecast 실행
submission = iterative_forecast_all(
    train_data=train_data,
    test_data_list=test_list,
    submission=submission,
    degrees=range(1, 11)
)


In [40]:
submission
# submission을 csv 파일로 저장합니다.
submission.to_csv('submission_filled.csv', index=False, encoding='utf-8-sig')
